# Claude Certified Architect — Foundations
## Domain 1: Agentic Architecture & Orchestration
**Exam weight: 27%**

Agentic systems fail in predictable ways: the loop exits too early, subagents have no context, tool gates rely on prompt instructions instead of code, and large reviews produce contradictory findings. This domain covers the foundational patterns that prevent those failures — correct loop control flow, coordinator-subagent architecture, programmatic enforcement, hook-based normalization, and decomposition strategies that determine whether a pipeline produces coherent output. At 27% exam weight, this is the domain where the most architectural judgment is tested.

This notebook walks through every task statement in Domain 1. Each section includes:
- What the concept means in practice
- Runnable code demonstrating the skill
- Anti-patterns alongside correct patterns where relevant

**Prerequisites:** `pip install anthropic`  
**Auth:** Set `ANTHROPIC_API_KEY` as an environment variable before running.

### Task Statements Covered
- **1.1** Design and implement agentic loops for autonomous task execution
- **1.2** Orchestrate multi-agent systems with coordinator-subagent patterns
- **1.3** Configure subagent invocation, context passing, and spawning
- **1.4** Implement multi-step workflows with enforcement and handoff patterns
- **1.5** Apply Agent SDK hooks for tool call interception and data normalization
- **1.6** Design task decomposition strategies for complex workflows
- **1.7** Manage session state, resumption, and forking

In [15]:
import anthropic
import json

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
print("Client ready.")

Client ready.


---
## Task Statement 1.1: Design and implement agentic loops for autonomous task execution

Every agent comes down to one loop: call the API, inspect `stop_reason`, execute tools if requested, append results, and call again. Getting the exit condition wrong — using iteration caps or text heuristics instead of `stop_reason == "end_turn"` — is the single most common cause of agents that terminate mid-task or spin indefinitely.

**What this means in practice:** Every agent you build needs a loop: send a request, inspect `stop_reason`, execute requested tools, append results, call again. The loop terminates exactly when Claude signals `"end_turn"` — not when an iteration counter expires, not when response text contains polite closing phrases. Until you see `"end_turn"`, there is still work to do.

**Why it matters for an architect:** This is the foundational control flow of all agentic systems. Getting it wrong — using iteration caps or text heuristics as primary stopping conditions — produces agents that terminate prematurely mid-task or spin indefinitely. Every multi-agent pattern in this certification builds on a correct agentic loop, so this is worth getting exactly right.


**Core concept:** After each API call, inspect `stop_reason`.  
- `"tool_use"` → execute the requested tools, append results, call again  
- `"end_turn"` → Claude is done; return the final response  

**Anti-patterns to avoid:**
- Parsing assistant text to decide whether to stop
- Using an iteration cap as the *primary* stopping mechanism
- Checking for text content as a completion indicator

In [2]:
# --- Shared tool definitions and stubs used throughout this notebook ---

TOOLS_1_1 = [
    {
        "name": "get_weather",
        "description": "Returns current weather for a city. Use when the user asks about weather conditions or temperature.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    },
    {
        "name": "get_population",
        "description": "Returns population for a city. Use when the user asks about how many people live in a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"]
        }
    }
]

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Stub tool executor. Replace with real backend calls in production."""
    if tool_name == "get_weather":
        return json.dumps({"city": tool_input["city"], "temp_f": 72, "condition": "sunny"})
    if tool_name == "get_population":
        return json.dumps({"city": tool_input["city"], "population": 2_720_546})
    return json.dumps({"error": f"unknown tool: {tool_name}"})

print("Tools and stubs defined.")

Tools and stubs defined.


In [3]:
# ANTI-PATTERN — do not use in production
# Primary stopping mechanism is an iteration cap + text heuristic.
# This breaks when Claude legitimately needs more iterations,
# or exits early when there are still tool calls pending.

def agentic_loop_antipattern(user_message: str, max_iterations: int = 5):
    messages = [{"role": "user", "content": user_message}]

    for i in range(max_iterations):  # BAD: cap as primary control
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=TOOLS_1_1, messages=messages
        )

        # BAD: parsing text content to decide whether to stop
        text = " ".join(b.text for b in response.content if hasattr(b, "text"))
        if "Let me know" in text or "I hope" in text:
            print(f"  [ANTI-PATTERN] Stopped on text heuristic at iteration {i+1}")
            return text

        if response.stop_reason == "end_turn":
            return text

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    print("  [ANTI-PATTERN] Hit iteration cap — may have stopped prematurely.")
    return None

print("Anti-pattern defined (for comparison only).")

Anti-pattern defined (for comparison only).


In [4]:
# CORRECT PATTERN
# Control flow is driven entirely by stop_reason. No caps. No text parsing.

def agentic_loop(user_message: str, tools: list = TOOLS_1_1, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": user_message}]
    iteration = 0

    while True:
        iteration += 1
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages
        )

        if verbose:
            print(f"  Iteration {iteration} | stop_reason: {response.stop_reason}")

        # Correct termination: model signals it is done
        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, "text"))

        # Continue: execute all requested tool calls and append results
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"    Tool call: {block.name}({block.input})")
                    result = execute_tool(block.name, block.input)
                    if verbose:
                        print(f"    Result:    {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Tool results accumulate in conversation history;
            # this is how the model reasons about the next action.
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        else:
            raise RuntimeError(f"Unexpected stop_reason: {response.stop_reason}")


# Run it
print("--- Correct agentic loop ---")
result = agentic_loop("What is the weather and population of Austin, Texas?")
print(f"\nFinal response:\n{result}")

--- Correct agentic loop ---
  Iteration 1 | stop_reason: tool_use
    Tool call: get_weather({'city': 'Austin, Texas'})
    Result:    {"city": "Austin, Texas", "temp_f": 72, "condition": "sunny"}
    Tool call: get_population({'city': 'Austin, Texas'})
    Result:    {"city": "Austin, Texas", "population": 2720546}
  Iteration 2 | stop_reason: end_turn

Final response:
Here's the information for **Austin, Texas**:

- 🌤️ **Weather:** Currently sunny with a temperature of **72°F**.
- 👥 **Population:** Approximately **2,720,546** people.

Austin enjoys pleasant weather today and is a thriving, populous city! Let me know if you'd like any more details.


**Observe:**
- The loop runs exactly as many iterations as Claude needs — no more, no less.
- Tool results are appended to `messages` before the next API call — this is how Claude retains tool output across iterations.
- Exit happens on `stop_reason == "end_turn"`, not on any text heuristic or cap.

**Key exam facts for 1.1:** `"tool_use"` → keep going. `"end_turn"` → stop. Never use caps as primary control.

---
## Task Statement 1.2: Orchestrate multi-agent systems with coordinator-subagent patterns

When a task is too complex for a single agent to handle reliably, the solution is not a longer prompt — it's a coordinator that owns routing and delegates to stateless subagents. The critical property of this pattern is that subagents have no memory of prior invocations and no visibility into the coordinator's history unless you explicitly pass it.

**What this means in practice:** In hub-and-spoke architecture, the coordinator owns all routing, error handling, and information flow. Subagents are stateless workers: they receive explicit context in their prompt, execute a task, and return a result. They have no memory of prior invocations and no visibility into the coordinator's conversation history unless you explicitly pass it.

**Why it matters for an architect:** This pattern makes multi-agent systems observable and debuggable — all information flow passes through the coordinator, so that's where you look when something goes wrong. The critical failure mode is narrow task decomposition: a coordinator that breaks "AI in creative industries" into only visual-arts subtasks will produce a pipeline where every agent executes perfectly yet the final report is incomplete.


**Core concept:** Hub-and-spoke architecture. A coordinator agent manages all inter-subagent communication, error handling, and routing. Subagents have **isolated context** — they do not automatically inherit the coordinator's conversation history. The coordinator decomposes tasks, delegates to subagents, aggregates results, and decides whether to iterate.

**Key risk:** Overly narrow task decomposition by the coordinator leads to incomplete coverage.

In [5]:
def run_subagent(system_prompt: str, user_prompt: str, tools: list = None) -> str:
    """
    Each subagent call is a fresh context.
    Subagents do NOT inherit the coordinator's conversation history —
    all necessary context must be passed explicitly in the prompt.
    """
    kwargs = {
        "model": MODEL,
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_prompt}]
    }
    if tools:
        kwargs["tools"] = tools
        # Use the agentic loop for tool-enabled subagents
        return agentic_loop_with_system(system_prompt, user_prompt, tools)

    response = client.messages.create(**kwargs)
    return " ".join(b.text for b in response.content if hasattr(b, "text"))


def agentic_loop_with_system(system: str, user_message: str, tools: list) -> str:
    """Agentic loop variant that accepts a system prompt (for subagents)."""
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system, tools=tools, messages=messages
        )
        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, "text"))
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})


def coordinator(topic: str) -> str:
    """
    Coordinator agent:
    1. Decomposes the topic into subtasks.
    2. Delegates each to a specialized subagent.
    3. Aggregates results.
    4. Synthesizes a final response.

    All subagent communication routes through the coordinator.
    """
    print(f"[Coordinator] Received topic: '{topic}'")

    # Step 1: Ask coordinator model to decompose the topic
    decomposition_prompt = f"""
You are a research coordinator. Break this topic into 2-3 distinct research subtopics
that together give broad coverage. Return them as a JSON list of strings.
Topic: {topic}
Return ONLY valid JSON, no explanation.
"""
    decomp_response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": decomposition_prompt}]
    )
    subtopics_raw = decomp_response.content[0].text.strip()
    # Strip markdown fences if present
    subtopics_raw = subtopics_raw.replace("```json", "").replace("```", "").strip()
    subtopics = json.loads(subtopics_raw)
    print(f"[Coordinator] Decomposed into subtopics: {subtopics}")

    # Step 2: Delegate each subtopic to a research subagent
    # Note: each subagent gets EXPLICIT context — it does not inherit coordinator state
    findings = {}
    for subtopic in subtopics:
        print(f"[Coordinator] Delegating to research subagent: '{subtopic}'")
        subagent_result = run_subagent(
            system_prompt="You are a concise research analyst. Summarize key facts on the given topic in 2-3 sentences.",
            # Explicit context: subagent gets the subtopic and the parent topic for framing
            user_prompt=f"Parent topic: {topic}\nYour assigned subtopic: {subtopic}\nProvide a 2-3 sentence summary."
        )
        findings[subtopic] = subagent_result
        print(f"[Coordinator] Received finding for '{subtopic}'")

    # Step 3: Coordinator synthesizes all findings
    findings_text = "\n\n".join(f"**{k}**:\n{v}" for k, v in findings.items())
    synthesis_prompt = f"""
You are synthesizing research findings into a coherent summary.
Topic: {topic}

Findings from subagents:
{findings_text}

Write a 3-4 sentence integrated summary covering all aspects.
"""
    synthesis_response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": synthesis_prompt}]
    )
    final = synthesis_response.content[0].text
    print("[Coordinator] Synthesis complete.")
    return final


result = coordinator("impact of AI on creative industries")
print(f"\n=== Final Report ===\n{result}")

[Coordinator] Received topic: 'impact of AI on creative industries'
[Coordinator] Decomposed into subtopics: ['AI as a creative tool: enhancing productivity and augmenting human creativity in art, music, and writing', 'Economic and labor market disruption: how AI automation affects employment, compensation, and ownership rights for creative professionals', 'Ethical, legal, and cultural implications: copyright concerns, authenticity debates, and the changing definition of creativity in an AI-driven landscape']
[Coordinator] Delegating to research subagent: 'AI as a creative tool: enhancing productivity and augmenting human creativity in art, music, and writing'
[Coordinator] Received finding for 'AI as a creative tool: enhancing productivity and augmenting human creativity in art, music, and writing'
[Coordinator] Delegating to research subagent: 'Economic and labor market disruption: how AI automation affects employment, compensation, and ownership rights for creative professionals'
[C

**Key exam facts for 1.2:**
- Each subagent receives its context explicitly in the prompt — there is no automatic inheritance.
- All communication routes through the coordinator, which maintains observability.
- The coordinator decomposes broadly; narrow decomposition (e.g., only visual arts for 'creative industries') is a key failure mode tested on the exam (see Question 7 in Phase 2).

---
## Task Statement 1.3: Configure subagent invocation, context passing, and spawning

Spawning a subagent is straightforward; getting it the right context is the hard part. In Claude Code, a coordinator emits `Task` tool calls — multiple in a single response for parallel execution — and every piece of context the subagent needs must be explicitly included in its prompt.

**What this means in practice:** To spawn a subagent, the coordinator emits a `Task` tool call — and `allowedTools` must include `"Task"` for this to be permitted. To spawn multiple subagents in parallel, emit multiple `Task` calls in a **single** coordinator response turn. Context for each subagent must be explicit in its prompt: web search results, prior analysis outputs, source URLs, page numbers — none of this is inherited automatically.

**Why it matters for an architect:** Parallel subagent spawning is the primary latency optimization in multi-agent pipelines — three agents in parallel takes as long as the slowest one; three sequential agents takes three times as long. Source attribution is also an architect's responsibility: structured context passing (claim + source URL + publication date) is the only way provenance survives the synthesis step.


**Core concepts:**
- In the Agent SDK, the `Task` tool is the mechanism for spawning subagents; `allowedTools` must include `"Task"`.
- Subagent context must be **explicitly provided** in the prompt.
- Parallel subagents are spawned by emitting **multiple Task tool calls in a single coordinator response**.
- Structured data formats should separate content from metadata (source URLs, page numbers) to preserve attribution.

The cell below models explicit context passing and structured output (the pattern that matters for the exam).

In [6]:
# Demonstrating: explicit context passing with structured attribution metadata
# This models what the Agent SDK's Task tool does — the subagent only knows
# what you put in its prompt.

# Simulated output from a web search subagent
web_search_findings = [
    {
        "claim": "Generative AI tools reduced music production time by up to 40% in 2024 studies.",
        "source_url": "https://example.com/ai-music-study",
        "publication_date": "2024-03"
    },
    {
        "claim": "AI image generators are used by over 30% of independent graphic designers.",
        "source_url": "https://example.com/design-survey",
        "publication_date": "2024-06"
    }
]

# Simulated output from a document analysis subagent
doc_analysis_findings = [
    {
        "claim": "Copyright frameworks have not kept pace with AI-generated content.",
        "source_document": "creative_economy_report.pdf",
        "page": 14,
        "publication_date": "2024-01"
    }
]

def pass_context_to_synthesis_subagent(
    web_findings: list,
    doc_findings: list,
    topic: str
) -> str:
    """
    Correct pattern: pass COMPLETE findings from prior subagents directly
    in the synthesis subagent's prompt, preserving source attribution.

    Structured format separates content (claims) from metadata (URLs, dates)
    so the synthesis agent can cite sources accurately.
    """
    # Serialize findings with full metadata — attribution must survive synthesis
    context_block = json.dumps({
        "web_search_findings": web_findings,
        "document_analysis_findings": doc_findings
    }, indent=2)

    synthesis_prompt = f"""
You are a synthesis agent. Your job is to combine research findings into a cited summary.

Topic: {topic}

Research findings (with source attribution):
{context_block}

Write a 3-4 sentence summary. For each claim, include the source URL or document name
and publication date in parentheses. Do not invent sources.
"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": synthesis_prompt}]
    )
    return response.content[0].text


synthesis_output = pass_context_to_synthesis_subagent(
    web_findings=web_search_findings,
    doc_findings=doc_analysis_findings,
    topic="impact of AI on creative industries"
)
print(synthesis_output)

AI is reshaping creative workflows at a significant pace, with generative AI tools reducing music production time by up to 40% according to recent research (https://example.com/ai-music-study, March 2024). Adoption is also widespread in visual design, as over 30% of independent graphic designers now incorporate AI image generators into their practice (https://example.com/design-survey, June 2024). However, this rapid integration has outpaced the legal landscape, as existing copyright frameworks have struggled to keep up with the complexities of AI-generated content (creative_economy_report.pdf, p. 14, January 2024). Together, these findings suggest that while AI is delivering measurable efficiency gains across creative disciplines, the industry faces unresolved regulatory challenges that will need to be addressed as adoption continues to grow.


### Parallel Subagent Spawning in Claude Code

In Claude Code, the coordinator emits `Task` tool calls to spawn subagents. **Parallel spawning requires all `Task` calls to appear in a single coordinator response turn** — emitting them in separate turns produces sequential execution.

**Parallel spawn — both subagents run concurrently:**

```json
// Single coordinator response turn — two Task calls = parallel execution
[
  {
    "type": "tool_use",
    "name": "Task",
    "input": {
      "description": "Research AI in music production",
      "prompt": "Parent topic: impact of AI on creative industries. Your subtopic: AI in music production. Summarize key facts in 2-3 sentences."
    }
  },
  {
    "type": "tool_use",
    "name": "Task",
    "input": {
      "description": "Research AI in visual arts",
      "prompt": "Parent topic: impact of AI on creative industries. Your subtopic: AI in visual arts. Summarize key facts in 2-3 sentences."
    }
  }
]
```

**Sequential spawn — two separate turns, each waits for the prior to finish:**

```json
// Turn 1 — single Task call
[{ "type": "tool_use", "name": "Task", "input": { "description": "Research music", "prompt": "..." } }]

// Turn 2 — second Task call after first completes
[{ "type": "tool_use", "name": "Task", "input": { "description": "Research visual arts", "prompt": "..." } }]
```

**Latency trade-off:** N parallel subagents take as long as the slowest one; N sequential subagents take the sum of all runtimes. Parallel spawning is the primary latency optimization in multi-agent pipelines.

**Structured output for attribution:** each subagent should return a JSON object that pairs content with source metadata (URL, publication date, document name). The coordinator passes these complete structured results to downstream subagents — attribution metadata must survive the synthesis step.

**Key exam facts for 1.3:**
- In the Agent SDK: `allowedTools` must include `"Task"` for a coordinator to spawn subagents.
- Subagents do not inherit parent context — pass it explicitly.
- Parallel spawning = multiple `Task` calls in a **single** coordinator response, not across separate turns.
- Use structured data to preserve attribution (source URLs, document names, dates) through synthesis.

---
## Task Statement 1.4: Implement multi-step workflows with enforcement and handoff patterns

When a business rule requires strict step ordering — verify identity before processing a refund, confirm eligibility before issuing a replacement — prompt instructions alone are not sufficient. The model's compliance is probabilistic; a programmatic gate that blocks downstream tool calls until prerequisites are met is deterministic.

**What this means in practice:** When business correctness requires step ordering — verify identity before processing a refund, confirm eligibility before issuing a replacement — use programmatic prerequisite gates, not prompt instructions. A gate blocks `process_refund` from executing until `get_customer` has returned a verified ID. Structured handoffs compile customer ID, root cause, and recommended action for human agents who have no access to the conversation transcript.

**Why it matters for an architect:** Prompt instructions are probabilistic. A 99% reliable instruction is a 1% failure rate in production — for financial operations, that is unacceptable. Programmatic gates are deterministic. The architect's job is to identify which steps require deterministic ordering and encode that in code, not in a system prompt.


**Core concept:** When business rules require a strict step order (e.g., verify identity before processing a refund), **prompt instructions alone have a non-zero failure rate**. Use programmatic prerequisites — gates that block downstream tool calls until prior steps complete — for deterministic compliance.

**Structured handoffs:** When escalating to a human, compile a structured summary (customer ID, root cause, recommended action) because the human agent does not have access to the conversation transcript.

In [8]:
# Demonstrating programmatic prerequisites
# The gate blocks process_refund until get_customer has returned a verified ID.

class WorkflowState:
    """Tracks which prerequisite steps have completed."""
    def __init__(self):
        self.verified_customer_id: str | None = None
        self.order_id: str | None = None

state = WorkflowState()

TOOLS_1_4 = [
    {
        "name": "get_customer",
        "description": "Looks up a customer by email or name. Must be called before any order or refund operations to verify identity.",
        "input_schema": {
            "type": "object",
            "properties": {"identifier": {"type": "string"}},
            "required": ["identifier"]
        }
    },
    {
        "name": "lookup_order",
        "description": "Retrieves order details by order ID. Requires a verified customer ID from get_customer first.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"]
        }
    },
    {
        "name": "process_refund",
        "description": "Processes a refund for a verified customer's order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "number"}
            },
            "required": ["order_id", "amount"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Escalates the case to a human agent with a structured handoff summary.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "root_cause": {"type": "string"},
                "recommended_action": {"type": "string"}
            },
            "required": ["customer_id", "root_cause", "recommended_action"]
        }
    }
]


def execute_tool_with_gate(tool_name: str, tool_input: dict, workflow_state: WorkflowState) -> str:
    """
    Programmatic gate: blocks process_refund and lookup_order
    until get_customer has returned a verified customer ID.

    This is deterministic enforcement — a prompt instruction to
    'always call get_customer first' would have a non-zero failure rate.
    """
    if tool_name == "get_customer":
        # Simulate customer lookup
        customer_id = "CUST-9921"
        workflow_state.verified_customer_id = customer_id  # Record completion
        return json.dumps({"customer_id": customer_id, "name": "Jane Smith", "email": tool_input["identifier"]})

    if tool_name == "lookup_order":
        # GATE: block if customer not yet verified
        if not workflow_state.verified_customer_id:
            return json.dumps({
                "error": "PREREQUISITE_FAILED",
                "message": "get_customer must be called and return a verified customer ID before lookup_order."
            })
        workflow_state.order_id = tool_input["order_id"]
        return json.dumps({"order_id": tool_input["order_id"], "amount": 49.99, "status": "delivered"})

    if tool_name == "process_refund":
        # GATE: block if customer not yet verified
        if not workflow_state.verified_customer_id:
            return json.dumps({
                "error": "PREREQUISITE_FAILED",
                "message": "get_customer must be called and verified before process_refund."
            })
        return json.dumps({"status": "success", "refund_id": "REF-4421", "amount": tool_input["amount"]})

    if tool_name == "escalate_to_human":
        # Structured handoff — human agent gets all context they need
        print(f"\n[ESCALATION HANDOFF]")
        print(f"  Customer ID:        {tool_input['customer_id']}")
        print(f"  Root Cause:         {tool_input['root_cause']}")
        print(f"  Recommended Action: {tool_input['recommended_action']}")
        return json.dumps({"status": "escalated", "ticket_id": "TKT-7734"})

    return json.dumps({"error": f"unknown tool: {tool_name}"})


def gated_agentic_loop(user_message: str, workflow_state: WorkflowState) -> str:
    messages = [{"role": "user", "content": user_message}]
    system = "You are a customer support agent. Always verify customer identity via get_customer before processing any orders or refunds."

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system,
            tools=TOOLS_1_4, messages=messages
        )

        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, "text"))

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool call: {block.name}({block.input})")
                    result = execute_tool_with_gate(block.name, block.input, workflow_state)
                    print(f"  Result:    {result}")
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})


ws = WorkflowState()
print("--- Gated workflow: refund request ---")
final = gated_agentic_loop(
    "Hi, I need a refund for order #ORD-5512. My email is jane@example.com.",
    ws
)
print(f"\nFinal response: {final}")
print(f"Verified customer ID recorded: {ws.verified_customer_id}")

--- Gated workflow: refund request ---
  Tool call: get_customer({'identifier': 'jane@example.com'})
  Result:    {"customer_id": "CUST-9921", "name": "Jane Smith", "email": "jane@example.com"}
  Tool call: lookup_order({'order_id': 'ORD-5512'})
  Result:    {"order_id": "ORD-5512", "amount": 49.99, "status": "delivered"}

Final response: Here are the details for your order:

- **Order ID:** ORD-5512
- **Amount:** $49.99
- **Status:** Delivered

Before I process the refund, could you let me know the **reason for the refund**? Also, would you like a **full refund of $49.99**, or a partial amount?
Verified customer ID recorded: CUST-9921


**Key exam facts for 1.4:** Prompt instructions for step ordering have a non-zero failure rate. When deterministic compliance is required (identity verification before financial operations), use **programmatic gates** that block downstream tools until prerequisites are met.

---
## Task Statement 1.5: Apply Agent SDK hooks for tool call interception and data normalization

Hooks intercept tool calls and their results at the infrastructure level, applying transformations and compliance rules before the model ever sees the data. This is the architectural answer to policy enforcement: deterministic guarantees that hold regardless of what the model reasons, applied once in a hook rather than repeated across every prompt.

**What this means in practice:** `PostToolUse` hooks transform tool outputs before the model processes them — normalizing Unix timestamps to ISO 8601, numeric status codes to strings, inconsistent field names to a common schema. Pre-call interception hooks enforce compliance rules before execution — blocking refunds above a dollar threshold and redirecting to a human escalation workflow.

**Why it matters for an architect:** Without hooks, you depend on the model consistently interpreting heterogeneous data formats and remembering policy limits — probabilistic compliance at best. Hooks give deterministic guarantees regardless of what the model reasons. They also centralize normalization logic: when a tool's output format changes, you update the hook, not every prompt that processes its output.


**Core concepts:**
- `PostToolUse` hooks intercept tool results **before the model processes them** — use for data normalization (e.g., converting Unix timestamps to ISO 8601).
- Tool call interception hooks intercept **outgoing** tool calls to enforce compliance rules (e.g., block refunds above a threshold).
- Hooks provide **deterministic guarantees**; prompt instructions provide **probabilistic compliance**.

We model both hook types explicitly since the Agent SDK's hook system runs server-side.

### Hooks in Claude Code: `PostToolUse` and `PreToolCall`

Hooks are shell scripts registered in `.claude/settings.json`. Claude Code invokes them at specific lifecycle points — no changes to tool executor code are needed.

#### Configuration

```json
// .claude/settings.json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "",
        "hooks": [{ "type": "command", "command": "~/.claude/hooks/normalize_output.sh" }]
      }
    ],
    "PreToolCall": [
      {
        "matcher": "process_refund",
        "hooks": [{ "type": "command", "command": "~/.claude/hooks/refund_gate.sh" }]
      }
    ]
  }
}
```

#### `PostToolUse` — normalize tool output before the model processes it

The hook receives raw tool result JSON on stdin and writes normalized JSON to stdout. Runs after every matching tool call, before the result is appended to conversation history.

```bash
#!/usr/bin/env bash
# ~/.claude/hooks/normalize_output.sh
# Converts Unix timestamps → ISO 8601 and numeric status codes → human-readable strings.
cat | python3 -c "
import json, sys, datetime
data = json.load(sys.stdin)
for field in ['created_at', 'updated_at']:
    if field in data and isinstance(data[field], int):
        data[field] = datetime.datetime.utcfromtimestamp(data[field]).strftime('%Y-%m-%dT%H:%M:%SZ')
STATUS_MAP = {1: 'active', 2: 'pending', 3: 'closed'}
if 'status_code' in data:
    data['status'] = STATUS_MAP.get(data['status_code'], 'unknown')
    del data['status_code']
print(json.dumps(data))
"
```

#### `PreToolCall` — intercept outgoing tool calls before execution

The hook receives the proposed tool call JSON on stdin. **Non-zero exit blocks the call.** Use for compliance rules that must fire regardless of what the model reasons.

```bash
#!/usr/bin/env bash
# ~/.claude/hooks/refund_gate.sh
# Blocks process_refund calls above $500 and surfaces a message for escalation.
AMOUNT=$(cat | python3 -c "import json,sys; print(json.load(sys.stdin).get('amount', 0))")
if (( $(echo "$AMOUNT > 500" | bc -l) )); then
  echo "BLOCKED: \$$AMOUNT exceeds \$500 threshold. Escalate to human." >&2
  exit 1
fi
```

**Why hooks instead of prompt instructions?** A prompt instruction like "only approve refunds under $500" has a non-zero failure rate — the model can reason incorrectly. A hook that exits non-zero is deterministic: it fires regardless of what the model decides. When a tool's output format changes, update the hook once; every system prompt that processes that output benefits automatically.

### Agent SDK Hooks: Python-Level Implementation

The exam appendix classifies `PostToolUse` and tool call interception as **Claude Agent SDK** concepts. In the Agent SDK, hooks are Python functions registered as callbacks — the same patterns as above, but implemented in code rather than bash scripts.

**Conceptual Python pattern for the same two hook types:**

```python
# PostToolUse hook: intercept tool results and normalize before the model processes them
def normalize_tool_output(tool_name: str, raw_result: dict) -> dict:
    """
    Runs after a tool executes, before the result is appended to conversation history.
    Converts Unix timestamps → ISO 8601, numeric status codes → human-readable strings.
    """
    import datetime
    for field in ["created_at", "updated_at"]:
        if field in raw_result and isinstance(raw_result[field], int):
            raw_result[field] = datetime.datetime.utcfromtimestamp(
                raw_result[field]
            ).strftime("%Y-%m-%dT%H:%M:%SZ")
    STATUS_MAP = {1: "active", 2: "pending", 3: "closed"}
    if "status_code" in raw_result:
        raw_result["status"] = STATUS_MAP.get(raw_result["status_code"], "unknown")
        del raw_result["status_code"]
    return raw_result


# Tool call interception hook: enforce compliance rules before execution
def enforce_refund_policy(tool_name: str, tool_input: dict) -> dict | None:
    """
    Runs before a tool executes. Return None to allow; return a block result to prevent.
    Blocks process_refund calls above $500 and redirects to human escalation.
    """
    if tool_name == "process_refund":
        amount = tool_input.get("amount", 0)
        if amount > 500:
            # Return a structured block result instead of executing the tool
            return {
                "blocked": True,
                "reason": f"Refund of ${amount:.2f} exceeds the $500 automated threshold.",
                "action": "call escalate_to_human with this context"
            }
    return None  # Allow the tool call to proceed
```

**Key distinction for the exam:**

| Hook context | Implementation |
|---|---|
| Claude Code (CLI) | Bash scripts in `.claude/settings.json` under `PostToolUse` / `PreToolCall` |
| Claude Agent SDK | Python callback functions registered with the agent at construction time |

Both patterns share the same architectural principle: **deterministic enforcement at the infrastructure level**, independent of what the model reasons. Hooks guarantee compliance; prompt instructions provide probabilistic compliance.

**Key exam facts for 1.5:** Use hooks (not prompt instructions) when business rules require **guaranteed** compliance. Hooks are deterministic; prompt instructions are probabilistic.

---
## Task Statement 1.6: Design task decomposition strategies for complex workflows

How you decompose a task determines output quality more than any prompt refinement. Single-pass review of many files produces attention dilution and contradictory findings; the right structure — per-file local analysis followed by a separate cross-file integration pass — is a decomposition decision, not a prompting one.

**What this means in practice:** Two decomposition strategies serve different work types. Prompt chaining (fixed sequential steps) handles predictable multi-aspect reviews: analyze each file for local issues first, then run a separate cross-file integration pass. Dynamic adaptive decomposition generates subtasks based on what is discovered at each step — the right approach for open-ended investigation where the shape of the work is unknown upfront.

**Why it matters for an architect:** Single-pass review of many files produces attention dilution — detailed feedback for some files, superficial comments for others, contradictory findings about the same pattern in different files. Per-file passes followed by a dedicated integration pass solve this structurally. Decomposition strategy determines output quality and cannot be compensated for with a better prompt.


**Core concept:** Two decomposition strategies:
- **Prompt chaining** — fixed sequential steps (e.g., per-file analysis, then cross-file integration pass). Use for predictable, multi-aspect reviews.
- **Dynamic adaptive decomposition** — generate subtasks based on what is discovered at each step. Use for open-ended investigation where structure is unknown upfront.

**Key technique:** Split large code reviews into per-file local analysis + a separate cross-file integration pass to avoid attention dilution.

In [11]:
# Demonstrating prompt chaining: per-file analysis followed by a cross-file integration pass.
# This avoids attention dilution that occurs when all files are reviewed in a single pass.

# Simulated code files for review
CODE_FILES = {
    "auth.py": """
def login(username, password):
    # TODO: add rate limiting
    user = db.query(f"SELECT * FROM users WHERE username='{username}'")
    if user and user.password == password:  # plaintext comparison
        return generate_token(user.id)
    return None
""",
    "orders.py": """
def process_order(user_id, order_data):
    # Assumes user is already verified upstream
    total = sum(item['price'] for item in order_data['items'])
    db.insert('orders', {'user_id': user_id, 'total': total})
    charge_customer(user_id, total)  # No error handling
    return {'status': 'ok'}
"""
}


def per_file_analysis_pass(files: dict) -> dict:
    """Pass 1: analyze each file individually for local issues."""
    findings = {}
    for filename, code in files.items():
        print(f"  Analyzing {filename}...")
        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            system="You are a security-focused code reviewer. Identify bugs and security issues in the provided file. Be specific and concise.",
            messages=[{"role": "user", "content": f"File: {filename}\n\n{code}"}]
        )
        findings[filename] = response.content[0].text
    return findings


def cross_file_integration_pass(files: dict, per_file_findings: dict) -> str:
    """Pass 2: separate integration pass focused on cross-file data flow issues."""
    files_block = "\n\n".join(f"=== {name} ===\n{code}" for name, code in files.items())
    findings_block = "\n\n".join(f"=== {name} findings ===\n{f}" for name, f in per_file_findings.items())

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system="You are a code reviewer focused on cross-file integration issues: trust boundary violations, inconsistent error handling across modules, data flow assumptions.",
        messages=[{"role": "user", "content": f"""
Code files:
{files_block}

Per-file findings from prior pass:
{findings_block}

Identify cross-file issues NOT already covered in the per-file findings.
"""}]
    )
    return response.content[0].text


print("=== Prompt Chaining: Multi-Pass Code Review ===")
print("\n--- Pass 1: Per-file local analysis ---")
per_file = per_file_analysis_pass(CODE_FILES)
for fname, finding in per_file.items():
    print(f"\n[{fname}]\n{finding}")

print("\n--- Pass 2: Cross-file integration analysis ---")
integration = cross_file_integration_pass(CODE_FILES, per_file)
print(integration)

=== Prompt Chaining: Multi-Pass Code Review ===

--- Pass 1: Per-file local analysis ---
  Analyzing auth.py...
  Analyzing orders.py...

[auth.py]
## Security Issues

### 1. SQL Injection (Critical)
```python
# Vulnerable
db.query(f"SELECT * FROM users WHERE username='{username}'")

# Fix: Use parameterized queries
db.query("SELECT * FROM users WHERE username = %s", (username,))
```
A malicious username like `' OR '1'='1` bypasses authentication entirely.

---

### 2. Plaintext Password Comparison (Critical)
```python
# Vulnerable
user.password == password

# Fix: Use a secure hashing library
import bcrypt
bcrypt.checkpw(password.encode(), user.hashed_password)
```
Passwords must never be stored or compared in plaintext. Use `bcrypt`, `argon2`, or `scrypt`.

---

### 3. Missing Rate Limiting (High)
The `# TODO` comment acknowledges this gap. Without rate limiting, the endpoint is vulnerable to **brute-force attacks**. Use a library like `slowapi` or enforce limits at the infrastructur

### Dynamic Adaptive Decomposition

**Contrast with prompt chaining:** Prompt chaining runs a fixed, predetermined sequence of steps — per-file analysis, then cross-file integration pass. The steps are the same regardless of what is found.

**Dynamic adaptive decomposition** generates the next subtask based on what was discovered in the prior step. The shape of the work is unknown upfront and emerges from the investigation itself. Use it for open-ended tasks where you cannot enumerate the steps in advance.

**Exam guide example:** *"Add comprehensive tests to a legacy codebase"* — the agent first maps structure, then identifies high-impact areas, then creates a prioritized plan that adapts as dependencies are discovered. The sequence of tool calls is not predetermined: discovering that `order_manager.py` depends on `payment_processor.py` changes which file should be tested first.

In [ ]:
# Dynamic adaptive decomposition: open-ended test planning for a legacy codebase.
# (Exam guide example: 'add comprehensive tests to a legacy codebase')
#
# The agent maps structure first, then lets each discovery drive the next investigation.
# Sequence of tool calls is NOT predetermined — it emerges from what is found.
# Contrast: prompt chaining would run a fixed per-file loop regardless of content.

TOOLS_ADAPTIVE = [
    {
        "name": "list_files",
        "description": "Lists source files in a directory. Use first to map overall codebase structure.",
        "input_schema": {
            "type": "object",
            "properties": {"directory": {"type": "string", "description": "Directory path to list"}},
            "required": ["directory"]
        }
    },
    {
        "name": "analyze_file",
        "description": "Analyzes a source file. Returns function count, external dependencies, complexity estimate, and whether tests already exist. Use to identify which files need the most testing work.",
        "input_schema": {
            "type": "object",
            "properties": {"file_path": {"type": "string"}},
            "required": ["file_path"]
        }
    },
    {
        "name": "check_test_coverage",
        "description": "Returns current test coverage percentage for a file. Use after analyze_file to quantify the testing gap.",
        "input_schema": {
            "type": "object",
            "properties": {"file_path": {"type": "string"}},
            "required": ["file_path"]
        }
    }
]

# Stub responses simulating a real legacy codebase.
# order_manager depends on payment_processor — a dependency the agent must discover
# and factor into its prioritization, not something hardcoded into the task prompt.
FILE_STUBS = {
    "src/": ["payment_processor.py", "auth.py", "order_manager.py", "utils.py", "email_notifier.py"],
    "payment_processor.py": {"functions": 12, "dependencies": ["stripe", "db", "auth"], "complexity": "high", "has_tests": False},
    "auth.py":               {"functions": 8,  "dependencies": ["jwt", "db"],             "complexity": "high", "has_tests": True},
    "order_manager.py":      {"functions": 15, "dependencies": ["payment_processor", "email_notifier", "db"], "complexity": "very high", "has_tests": False},
    "utils.py":              {"functions": 6,  "dependencies": [],                        "complexity": "low",  "has_tests": True},
    "email_notifier.py":     {"functions": 4,  "dependencies": ["smtp"],                  "complexity": "medium", "has_tests": False},
}
COVERAGE_STUBS = {
    "payment_processor.py": 0,
    "auth.py":              62,
    "order_manager.py":     0,
    "utils.py":             88,
    "email_notifier.py":    15,
}

def adaptive_tool_executor(tool_name: str, tool_input: dict) -> str:
    if tool_name == "list_files":
        files = FILE_STUBS.get(tool_input["directory"], [])
        return json.dumps({"directory": tool_input["directory"], "files": files})
    if tool_name == "analyze_file":
        name = tool_input["file_path"].split("/")[-1]
        return json.dumps({"file": tool_input["file_path"], **FILE_STUBS.get(name, {"error": "not found"})})
    if tool_name == "check_test_coverage":
        name = tool_input["file_path"].split("/")[-1]
        return json.dumps({"file": tool_input["file_path"], "coverage_pct": COVERAGE_STUBS.get(name, 0)})
    return json.dumps({"error": f"unknown tool: {tool_name}"})


def adaptive_decomposition_agent(task: str) -> str:
    """
    Dynamic adaptive decomposition.
    The agent decides what to investigate next at each step based on prior discoveries.
    No predetermined sequence — the plan emerges from the investigation.
    """
    messages = [{"role": "user", "content": task}]
    system = (
        "You are a test planning agent for a legacy codebase. Investigate adaptively:\n"
        "1. Start by mapping overall structure.\n"
        "2. Based on what you find, decide which files are highest-impact — do not analyze all files uniformly.\n"
        "3. Let dependency discoveries change your priority order (e.g., if B depends on A, test A first).\n"
        "4. Produce a prioritized test plan only after your discoveries justify it.\n"
        "Think explicitly about what to investigate next and why before each tool call."
    )

    step = 0
    while True:
        step += 1
        response = client.messages.create(
            model=MODEL, max_tokens=1500, system=system,
            tools=TOOLS_ADAPTIVE, messages=messages
        )
        print(f"  Step {step} | stop_reason: {response.stop_reason}")

        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, "text"))

        if response.stop_reason == "tool_use":
            # Print any reasoning text the model emits before tool calls
            for block in response.content:
                if hasattr(block, "text") and block.text.strip():
                    print(f"    [reasoning] {block.text.strip()[:120]}")

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"    -> {block.name}({block.input})")
                    result = adaptive_tool_executor(block.name, block.input)
                    print(f"       {result}")
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})


print("=== Dynamic Adaptive Decomposition: Legacy Codebase Test Plan ===\n")
plan = adaptive_decomposition_agent(
    "Add comprehensive tests to our legacy codebase in src/. "
    "Build a prioritized test plan by mapping structure, identifying high-impact areas, "
    "and accounting for dependencies between modules as you discover them."
)
print(f"\n=== Prioritized Test Plan ===\n{plan}")


**Key exam facts for 1.6:** Two decomposition patterns:
- **Prompt chaining** → predictable, multi-aspect reviews (per-file + cross-file passes)
- **Dynamic adaptive decomposition** → open-ended investigation (generate subtasks based on discoveries)

Splitting into passes avoids **attention dilution** and **contradictory findings** that occur in single large-context reviews.

---
## Task Statement 1.7: Manage session state, resumption, and forking

Not every multi-session workflow should resume from where it left off — when code has changed between sessions, prior tool results describe old state and resuming produces confident but incorrect reasoning. Understanding when to resume, when to start fresh with an injected summary, and when to fork for divergent exploration is the judgment call this task statement tests.

**What this means in practice:** `--resume <session-name>` continues a named prior session when that context is still valid. `fork_session` creates independent branches from a shared analysis baseline — use it to explore two refactoring approaches or two testing strategies without results contaminating each other. When prior tool results are stale (code has changed since the session), starting fresh with a structured summary injected as context is more reliable than resuming.

**Why it matters for an architect:** Resuming with stale tool results means the agent reasons from incorrect state — errors are subtle and invisible until they surface in production. Understanding when to resume vs. inject a fresh summary vs. fork is the judgment call the exam tests through scenario questions about what changed between sessions.


**Core concepts:**
- `--resume <session-name>` continues a named prior session.
- `fork_session` creates independent branches from a shared analysis baseline.
- When resuming after code changes, inform the agent about which files changed — don't require full re-exploration.
- When prior tool results are stale, start a **new session with a structured summary** injected as context rather than resuming.

We model the pattern of structured summary injection (the core skill being tested).

### Session Management in Claude Code: `--resume` and `fork_session`

#### Starting and resuming named sessions

```bash
# Start a session with a name you can resume later
claude --session payment-service-audit

# Resume when prior context is still valid (context, tool results, and history are intact)
claude --resume payment-service-audit
```

#### Using `fork_session` to explore divergent approaches without cross-contamination

`fork_session` creates an independent branch from the current session's analysis baseline. In Claude Code, it is invoked with the `/fork` slash command from within a running session:

```
# Inside a running Claude Code session
/fork strategy-integration-tests
# Changes and tool calls in this fork do not affect the parent session
```

Use `fork_session` when you want to explore two approaches (e.g., integration tests vs. unit tests) from a shared analysis baseline without the branches influencing each other.

#### When to start fresh instead of resuming

If source files have changed since the prior session, the existing tool results describe old code. Resuming causes the model to reason from stale state — errors are confident but incorrect. Instead, start a new session and inject a structured summary:

```bash
# Inject prior session findings as context for the new session
claude --print "$(cat prior_session_summary.md)"
```

The summary should explicitly name which files changed so the agent focuses re-analysis on only those files rather than re-exploring the entire codebase.

**Example summary structure:**
```json
{
  "codebase": "payment-service",
  "architecture_findings": [
    "Entry point: src/main.py → PaymentRouter → PaymentProcessor",
    "Auth middleware: src/middleware/auth.py validates JWT on every request"
  ],
  "high_risk_areas": ["src/refund.py — no idempotency keys", "src/webhook.py — no signature verification"],
  "files_changed_since_session": ["src/refund.py"]
}
```

**Decision table:**

| Situation | Approach |
|-----------|----------|
| Continuing an investigation; context still valid | `claude --resume session-name` |
| Exploring two strategies without cross-contamination | `fork_session` (invoked as `/fork <branch-name>` in Claude Code) |
| Files changed since last session; tool results stale | New session + inject structured summary; name changed files explicitly |

**Key exam facts for 1.7:**
- `--resume <session-name>` → continue a named session (when prior context is mostly valid)
- `fork_session` → divergent exploration from a shared baseline (branches don't cross-contaminate); invoked as `/fork <branch-name>` in Claude Code
- After code changes: inform the resumed session about **which files changed** for targeted re-analysis
- When tool results are stale: **start fresh + inject a structured summary** rather than resuming

---
## Domain 1 Capstone Project: Customer Support Resolution Agent

**Description:** Build a support agent that handles return and billing requests using the patterns from this domain:
- Correct agentic loop (1.1)
- Programmatic prerequisites: verify customer before refund (1.4)
- Pre-call hook: block refunds above $500 (1.5)
- PostToolUse hook: normalize tool output formats (1.5)
- Structured escalation handoff (1.4)

**Success criteria:** The agent correctly verifies the customer, processes eligible refunds, blocks over-threshold refunds with a structured escalation, and terminates cleanly on `end_turn`.

In [ ]:
# Domain 1 Capstone: Customer Support Resolution Agent
#
# Demonstrates:
# - Correct agentic loop (1.1): loop on stop_reason, not iteration caps
# - Programmatic prerequisites: verify customer before any order/refund operation (1.4)
# - Structured escalation handoff (1.4): human agent receives customer_id, root_cause, action
#
# In a Claude Code deployment, the $500 refund threshold (PreToolCall) and output
# normalization (PostToolUse) would be enforced by hook scripts in .claude/settings.json
# rather than inline Python — see Task 1.5 for that configuration.

REFUND_THRESHOLD = 500.00

ALL_TOOLS = [
    {
        "name": "get_customer",
        "description": "Looks up and verifies a customer by email. Must be called before any order or refund operations.",
        "input_schema": {"type": "object", "properties": {"email": {"type": "string"}}, "required": ["email"]}
    },
    {
        "name": "lookup_order",
        "description": "Retrieves order details by order ID. Requires prior customer verification.",
        "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}
    },
    {
        "name": "process_refund",
        "description": "Processes a refund. Requires verified customer. Amounts over $500 will be escalated.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}},
            "required": ["order_id", "amount"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Escalates to a human agent with a structured handoff summary.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "root_cause": {"type": "string"},
                "recommended_action": {"type": "string"}
            },
            "required": ["customer_id", "root_cause", "recommended_action"]
        }
    }
]


def capstone_tool_executor(tool_name: str, tool_input: dict, state: WorkflowState) -> str:
    if tool_name == "get_customer":
        state.verified_customer_id = "CUST-9921"
        return json.dumps({
            "customer_id": "CUST-9921",
            "name": "Jane Smith",
            "email": tool_input["email"],
            "created_at": "2024-01-01T00:00:00Z",
            "status": "active"
        })

    if tool_name == "lookup_order":
        if not state.verified_customer_id:
            return json.dumps({"error": "PREREQUISITE_FAILED: call get_customer first"})
        return json.dumps({"order_id": tool_input["order_id"], "amount": 749.99, "status": "delivered"})

    if tool_name == "process_refund":
        if not state.verified_customer_id:
            return json.dumps({"error": "PREREQUISITE_FAILED: call get_customer first"})
        amount = tool_input.get("amount", 0)
        if amount > REFUND_THRESHOLD:
            return json.dumps({
                "blocked": True,
                "reason": f"Refund of ${amount:.2f} exceeds the ${REFUND_THRESHOLD:.2f} automated threshold.",
                "suggest": "call escalate_to_human"
            })
        return json.dumps({"status": "success", "refund_id": "REF-0001", "amount": amount})

    if tool_name == "escalate_to_human":
        print(f"\n    [ESCALATION] Customer: {tool_input['customer_id']}")
        print(f"    [ESCALATION] Cause: {tool_input['root_cause']}")
        print(f"    [ESCALATION] Action: {tool_input['recommended_action']}")
        return json.dumps({"ticket_id": "TKT-8841", "status": "escalated"})

    return json.dumps({"error": f"unknown tool: {tool_name}"})


def run_support_agent(user_message: str):
    print(f"\n{'='*60}")
    print(f"Customer: {user_message}")
    print('='*60)

    state = WorkflowState()
    messages = [{"role": "user", "content": user_message}]
    system = (
        "You are a customer support agent. Always verify the customer via get_customer before "
        "any order or refund operations. For refunds blocked by policy, escalate with a structured summary."
    )

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system, tools=ALL_TOOLS, messages=messages
        )
        print(f"  [stop_reason: {response.stop_reason}]")

        if response.stop_reason == "end_turn":
            final = " ".join(b.text for b in response.content if hasattr(b, "text"))
            print(f"\nAgent response: {final}")
            return

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool: {block.name}({block.input})")
                    result = capstone_tool_executor(block.name, block.input, state)
                    print(f"  Result: {result}")
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})


run_support_agent("Hi, I need to return order #ORD-5512. My email is jane@example.com.")

---
## Domain 1 Complete

**Summary of key exam facts:**

| Task | Core Principle |
|------|----------------|
| 1.1 | `stop_reason == "tool_use"` → continue; `"end_turn"` → stop. No caps or text heuristics. |
| 1.2 | Hub-and-spoke: coordinator manages all routing. Subagents have isolated context. |
| 1.3 | Subagent context is explicit. Parallel = multiple Task calls in one coordinator response. |
| 1.4 | Programmatic gates for deterministic step ordering. Structured handoffs for escalation. |
| 1.5 | PostToolUse hooks normalize output. Pre-call hooks enforce compliance. Hooks > prompts for guarantees. |
| 1.6 | Prompt chaining for predictable reviews. Dynamic decomposition for open-ended investigation. |
| 1.7 | Resume when context is valid. Fresh + injected summary when tool results are stale. fork_session for divergent branches. |